# ER-CyRIS Siklus 1 — STEP 2A
## S0 Baseline Supervised (XGBoost + Random Forest) pada 4 Dataset

Fokus tahap ini hanya **S0 supervised**.

Belum menjalankan Isolation Forest, LOF, DBSCAN, S1–S5, atau SHAP.

Dataset:
1. CICIDS2018
2. HDFS (representasi richer 22 fitur yang sudah lolos audit konflik label)
3. BGL
4. UNSW-NB15

Desain mempertahankan inti eksperimen Siklus 1 asli:
- split 80/20,
- XGBoost,
- Random Forest,
- Precision, Recall, F1, PR-AUC, FAR, Alert Rate.

Perbaikan metodologis:
- preprocessing di-fit hanya pada training,
- feature-vector identik tidak boleh tersebar antara train dan test,
- HDFS memakai representasi 22 fitur yang sudah diverifikasi,
- UNSW-NB15 dibaca sebagai file 49 kolom tanpa header,
- hasil disimpan terpisah sebagai confirmatory multi-dataset replication.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, zlib, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print("✅ Environment ready")

In [ ]:
BASE = '/content/drive/MyDrive/ercyris_siklus2/'
OUT = Path('/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/')
OUT.mkdir(parents=True, exist_ok=True)

CFG = {
    'CICIDS': BASE + '02-14-2018.csv',
    'BGL': BASE + 'BGL.log',
    'UNSW': BASE + 'UNSW-NB15_1.csv',
    'HDFS_CACHE': (
        '/content/drive/MyDrive/ercyris_siklus1_confirmatory/cache/'
        'HDFS_richer_22f_confirmatory.parquet'
    ),
    'MAX_CICIDS': 500_000,
    'MAX_BGL': 50_000,
    'TEST_SIZE': 0.20,
    'SEED': 42,
}

for k in ['CICIDS','BGL','UNSW','HDFS_CACHE']:
    p = CFG[k]
    print(f"{k:12s} {'✅' if os.path.exists(p) else '❌'} {p}")

if not os.path.exists(CFG['HDFS_CACHE']):
    raise FileNotFoundError(
        "HDFS cache belum ditemukan. Kembali ke notebook STEP 1 dan simpan "
        "X_hdfs2 + y_hdfs2 ke HDFS_richer_22f_confirmatory.parquet."
    )

In [ ]:
def clean_cols(cols):
    out = []
    for c in cols:
        c = str(c).strip().lower().replace(' ', '_').replace('/', '_per_')
        c = re.sub(r'[^a-z0-9_]+', '_', c).strip('_')
        out.append(c)
    return out

def stable_hash_1000(text):
    return zlib.crc32(str(text).encode('utf-8')) % 1000

def feature_groups(X):
    return pd.util.hash_pandas_object(
        X.reset_index(drop=True),
        index=False
    ).astype('uint64').values

def split_no_duplicate_leakage(X, y, test_size=0.20, seed=42):
    y = np.asarray(y, dtype=int)
    groups = feature_groups(X)
    n_unique = len(np.unique(groups))

    if n_unique == len(X):
        idx = np.arange(len(X))
        tr, te = train_test_split(
            idx,
            test_size=test_size,
            random_state=seed,
            stratify=y
        )
        method = 'stratified_unique'
    else:
        best = None
        overall_rate = y.mean()

        for rs in range(seed, seed + 200):
            gss = GroupShuffleSplit(
                n_splits=1,
                test_size=test_size,
                random_state=rs
            )
            tr0, te0 = next(gss.split(X, y, groups))

            if len(np.unique(y[tr0])) < 2 or len(np.unique(y[te0])) < 2:
                continue

            test_frac = len(te0) / len(X)
            test_rate = y[te0].mean()

            score = (
                abs(test_frac - test_size)
                + 2.0 * abs(test_rate - overall_rate)
            )

            if best is None or score < best[0]:
                best = (score, tr0, te0, rs)

        if best is None:
            raise RuntimeError("Tidak ditemukan group-aware split yang valid.")

        _, tr, te, chosen_seed = best
        method = f'group_aware_seed_{chosen_seed}'

        shared = np.intersect1d(
            np.unique(groups[tr]),
            np.unique(groups[te])
        )
        assert len(shared) == 0, "Feature-group leakage masih terjadi."

    audit = {
        'method': method,
        'n_total': len(X),
        'n_train': len(tr),
        'n_test': len(te),
        'train_pos_rate': float(y[tr].mean()),
        'test_pos_rate': float(y[te].mean()),
        'overall_pos_rate': float(y.mean()),
        'unique_feature_vectors': int(n_unique),
        'duplicate_rows': int(len(X) - n_unique),
    }
    return np.asarray(tr), np.asarray(te), audit

def make_preprocessor(Xtr):
    numeric_cols = list(Xtr.select_dtypes(include=np.number).columns)
    categorical_cols = [c for c in Xtr.columns if c not in numeric_cols]

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ])

    pre = ColumnTransformer(
        transformers=[
            ('num', num_pipe, numeric_cols),
            ('cat', cat_pipe, categorical_cols),
        ],
        remainder='drop',
        sparse_threshold=0
    )

    pipe = Pipeline([
        ('columns', pre),
        ('scale', MinMaxScaler())
    ])
    return pipe

def calc_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0,1]
    ).ravel()

    N = tn + fp + fn + tp

    return {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'PR_AUC': average_precision_score(y_true, y_score),
        'FAR': fp / max(fp + tn, 1),
        'Alert_Rate': (tp + fp) / max(N, 1),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp),
    }

In [ ]:
def load_cicids():
    print("\n📂 CICIDS2018")

    df = pd.read_csv(CFG['CICIDS'], low_memory=False)

    if len(df) > CFG['MAX_CICIDS']:
        df = df.sample(
            n=CFG['MAX_CICIDS'],
            random_state=CFG['SEED']
        )

    df.columns = clean_cols(df.columns)

    label_col = next(c for c in df.columns if 'label' in c)
    y = (
        df[label_col].astype(str).str.strip().str.upper()
        != 'BENIGN'
    ).astype(int).values

    drop = [label_col] + [
        c for c in df.columns
        if any(k in c for k in [
            'timestamp','flow_id',
            'src_ip','dst_ip',
            'source_ip','destination_ip'
        ])
    ]

    X = (
        df.drop(columns=list(set(drop)), errors='ignore')
        .replace([np.inf,-np.inf], np.nan)
        .reset_index(drop=True)
    )

    tmp = X.copy()
    tmp['__label__'] = y
    feat = list(X.columns)

    before = len(tmp)
    tmp = tmp.drop_duplicates(
        subset=feat,
        keep='first'
    ).reset_index(drop=True)

    y = tmp.pop('__label__').astype(int).values
    X = tmp

    print(
        f"   {before:,} -> {len(X):,} rows | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X, y

In [ ]:
def load_hdfs():
    print("\n📂 HDFS richer 22f")

    df = pd.read_parquet(CFG['HDFS_CACHE'])

    if '__label__' not in df.columns:
        raise ValueError("Kolom __label__ tidak ditemukan pada HDFS cache.")

    y = df.pop('__label__').astype(int).values
    X = df.reset_index(drop=True)

    print(
        f"   {len(X):,} rows | {X.shape[1]} features | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X, y

In [ ]:
def load_bgl():
    print("\n📂 BGL")

    SM = {
        'INFO':1,'DEBUG':0,'WARN':2,'WARNING':2,'ERROR':3,'SEVERE':4,
        'FATAL':4,'APPFPO':3,'KERN':2,'KERNSVC':3,'APPFPW':2
    }

    AKW = [
        'fatal','error','fail','exception','abort','illegal',
        'segfault','panic','crash','out of memory','overflow'
    ]

    rows = []

    with open(CFG['BGL'], 'r', errors='replace') as f:
        for seq,line in enumerate(tqdm(f, desc='parse BGL', leave=False)):
            p = line.split(None, 8)
            if len(p) < 9:
                continue

            label = 0 if p[0] == '-' else 1
            sev = SM.get(p[6].upper(),1)
            msg = p[8].lower()

            try:
                hour = int(p[3].split(':')[0]) if ':' in p[3] else 12
            except:
                hour = 12

            rows.append({
                'sev': sev,
                'is_err': int(sev >= 3),
                'is_warn': int(sev == 2),
                'hour': hour,
                'off_hr': int(hour < 6),
                'peak_hr': int(8 <= hour <= 18),
                'err_kw': int(any(k in msg for k in AKW)),
                'has_num': int(bool(re.search(r'\b\d{4,}\b', msg))),
                'msg_len': min(len(msg),500),
                'comp_len': len(p[5]),
                'node_h': stable_hash_1000(p[4]),
                '__label__': label,
                '__seq__': seq
            })

    df = pd.DataFrame(rows)

    if len(df) > CFG['MAX_BGL']:
        selected = []
        for cls in [0,1]:
            g = df[df['__label__'] == cls]
            n = int(round(
                CFG['MAX_BGL'] * len(g) / len(df)
            ))
            selected.append(
                g.sample(
                    n=min(n,len(g)),
                    random_state=CFG['SEED']
                )
            )

        df = (
            pd.concat(selected)
            .sort_values('__seq__')
            .reset_index(drop=True)
        )

    y = df.pop('__label__').astype(int).values
    df.drop(columns=['__seq__'], inplace=True)
    X = df

    print(
        f"   {len(X):,} rows | {X.shape[1]} features | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X, y

In [ ]:
UNSW_COLUMNS = [
    'srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
    'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
    'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
    'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
    'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
    'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
    'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
    'ct_dst_src_ltm','attack_cat','label'
]

def load_unsw():
    print("\n📂 UNSW-NB15")

    raw = pd.read_csv(
        CFG['UNSW'],
        header=None,
        low_memory=False
    )

    if raw.shape[1] != len(UNSW_COLUMNS):
        raise ValueError(
            f"UNSW columns={raw.shape[1]}, expected={len(UNSW_COLUMNS)}"
        )

    raw.columns = UNSW_COLUMNS

    y = (
        pd.to_numeric(raw['label'], errors='coerce')
        .fillna(0)
        .astype(int)
        .values
    )

    X = raw.drop(
        columns=[
            'label','attack_cat',
            'srcip','dstip',
            'stime','ltime'
        ],
        errors='ignore'
    ).copy()

    X.columns = clean_cols(X.columns)
    X.replace([np.inf,-np.inf], np.nan, inplace=True)

    print(
        f"   {len(X):,} rows | {X.shape[1]} features | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X.reset_index(drop=True), y

In [ ]:
DATASETS = {
    'CICIDS2018': load_cicids(),
    'HDFS': load_hdfs(),
    'BGL': load_bgl(),
    'UNSW-NB15': load_unsw(),
}

print("\n✅ All four datasets loaded.")

## Split Audit

Sebelum modeling, notebook mencatat:
- jumlah train/test,
- prevalensi positive train/test,
- jumlah feature vector unik,
- duplicate rows,
- metode split.

Untuk HDFS, BGL, dan UNSW-NB15, feature vector identik tidak boleh muncul sekaligus pada train dan test.

In [ ]:
SPLITS = {}
split_rows = []

for name,(X,y) in DATASETS.items():
    print(f"\n🔀 Splitting {name} ...")

    tr, te, audit = split_no_duplicate_leakage(
        X, y,
        test_size=CFG['TEST_SIZE'],
        seed=CFG['SEED']
    )

    SPLITS[name] = (tr,te)
    split_rows.append({'dataset': name, **audit})
    print(audit)

split_audit_df = pd.DataFrame(split_rows)
display(split_audit_df)

split_audit_df.to_csv(
    OUT / 'S0_split_audit.csv',
    index=False
)

print("✅ Split audit saved.")

## S0 Supervised Baseline

Model:
- XGBoost
- Random Forest

Keduanya dilatih pada seluruh training partition dan dievaluasi pada test partition yang sudah bebas feature-group leakage.

In [ ]:
RESULTS = []

for name,(X,y) in DATASETS.items():

    print(f"\n{'='*74}")
    print(f"S0 SUPERVISED — {name}")
    print('='*74)

    tr,te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()
    ytr = y[tr]
    yte = y[te]

    pre = make_preprocessor(Xtr_raw)

    t0 = time.perf_counter()
    Xtr = pre.fit_transform(Xtr_raw)
    Xte = pre.transform(Xte_raw)
    prep_sec = time.perf_counter() - t0

    print(
        f"Preprocessed: train={Xtr.shape}, test={Xte.shape}, "
        f"time={prep_sec:.1f}s"
    )

    models = {
        'XGBoost': XGBClassifier(
            random_state=CFG['SEED'],
            n_jobs=-1,
            eval_metric='logloss',
            tree_method='hist',
            verbosity=0
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100,
            random_state=CFG['SEED'],
            n_jobs=-1
        )
    }

    for model_name, model in models.items():

        print(f"  ▶ {model_name}")

        t0 = time.perf_counter()
        model.fit(Xtr, ytr)
        train_sec = time.perf_counter() - t0

        t1 = time.perf_counter()
        y_pred = model.predict(Xte)
        y_score = model.predict_proba(Xte)[:,1]
        infer_sec = time.perf_counter() - t1

        m = calc_metrics(yte, y_pred, y_score)

        RESULTS.append({
            'dataset': name,
            'scenario': 'S0',
            'model': model_name,
            'n_train': len(tr),
            'n_test': len(te),
            'train_positive_rate': float(ytr.mean()),
            'test_positive_rate': float(yte.mean()),
            'preprocessing_sec': prep_sec,
            'train_sec': train_sec,
            'inference_sec': infer_sec,
            **m
        })

        print(
            f"     F1={m['F1']:.6f} | "
            f"Precision={m['Precision']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f}"
        )

results_df = pd.DataFrame(RESULTS)
display(results_df)

results_df.to_csv(
    OUT / 'S0_supervised_results.csv',
    index=False
)

print("\n✅ STEP 2A completed and saved:")
print(OUT / 'S0_supervised_results.csv')

## STOP

Jangan lanjut ke model unsupervised atau S1–S5.

Kirim ke ChatGPT dua tabel:
1. `S0_split_audit.csv`
2. `S0_supervised_results.csv`

Setelah diperiksa, baru dilanjutkan ke STEP 2B: Isolation Forest, LOF, dan DBSCAN.